# 02 - Rebuild problems.parquet with real descriptions
Reads the CodeNet HTML statements for our 800 problems, converts them to clean text, adds title and limits, and reports coverage.
The old file is backed up first.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
from paths import *
print("repo root:", ROOT)

repo root: C:\Users\yassi\OneDrive\Documents\GitHub\WhoWroteIt-


In [2]:
import re
import pandas as pd
from bs4 import BeautifulSoup

old = pd.read_parquet(INTERIM / "problems.parquet")
print("old columns:", old.columns.tolist(), "| rows:", len(old))
ID = "problem_id" if "problem_id" in old.columns else old.columns[0]
ids = old[ID].astype(str).str.strip().tolist()
print("example ids:", ids[:5])

html_dir = find_dir(CODENET_DIR, "problem_descriptions")
plist = find_file(CODENET_DIR, "problem_list.csv")
assert html_dir is not None, "problem_descriptions folder not found under data/external/codenet"
assert plist is not None, "problem_list.csv not found under data/external/codenet"

meta = pd.read_csv(plist)
print("problem_list columns:", meta.columns.tolist())
id_col = "id" if "id" in meta.columns else meta.columns[0]
meta[id_col] = meta[id_col].astype(str).str.strip()
meta = meta.set_index(id_col)

old columns: ['problem_id', 'title', 'description', 'n_chars', 'has_description', 'in_aigcodeset_problem'] | rows: 800
example ids: ['p00000', 'p00001', 'p00002', 'p00003', 'p00005']
problem_list columns: ['id', 'name', 'dataset', 'time_limit', 'memory_limit', 'rating', 'tags', 'complexity']


In [3]:
def html_to_text(path):
    html = path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")
    for t in soup(["script", "style"]):
        t.decompose()
    tag = soup.find(["h1", "title"])
    title = tag.get_text(" ", strip=True) if tag else ""
    text = soup.get_text("\n", strip=True)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return title, text

html_index = {p.stem: p for p in html_dir.rglob("*.html")}
rows = []
for pid in ids:
    path = html_index.get(pid)
    title, text = ("", "")
    if path is not None:
        title, text = html_to_text(path)
    m = meta.loc[pid] if pid in meta.index else None
    if m is not None and "name" in meta.columns and isinstance(m["name"], str) and m["name"].strip():
        title = m["name"].strip()
    rows.append({
        ID: pid,
        "title": title,
        "description": text,
        "time_limit_ms": m["time_limit"] if m is not None and "time_limit" in meta.columns else None,
        "memory_limit_kb": m["memory_limit"] if m is not None and "memory_limit" in meta.columns else None,
    })

new = pd.DataFrame(rows)
has_desc = new["description"].str.len() >= 50
print(f"Coverage: {int(has_desc.sum())} of {len(new)} problems have a description")
missing = new.loc[~has_desc, ID].tolist()
print("Missing:", missing[:30], "..." if len(missing) > 30 else "")

Coverage: 800 of 800 problems have a description
Missing: [] 


In [4]:
display(new.head(3))
ex = new[has_desc].iloc[0]
print("\n--- example:", ex[ID], "|", ex["title"], "---")
print(ex["description"][:800])

,problem_id,title,description,time_limit_ms,memory_limit_kb
0,p00000,QQ,QQ\nWrite a program which prints multiplicatio...,1000.0,131072.0
1,p00001,List of Top 3 Hills,List of Top 3 Hills\nThere is a data which pro...,1000.0,131072.0
2,p00002,Digit Number,Digit Number\nWrite a program which computes t...,1000.0,131072.0



--- example: p00000 | QQ ---
QQ
Write a program which prints multiplication tables in the following format:
1x1=1
1x2=2
.
.
9x8=72
9x9=81
Input
No input.
Output
1x1=1
1x2=2
.
.
9x8=72
9x9=81
Template for C
#include<stdio.h>

int main(){

 return 0;
}
Template for C++
#include<iostream>
using namespace std;

int main(){

 return 0;
}
Template for Java
class Main{
 public static void main(String[] a){

 }
}


In [5]:
# Keep every old column, replace/add the new ones. Back up the old file once.
backup = INTERIM / "problems_before_descriptions.parquet"
if not backup.exists():
    old.to_parquet(backup, index=False)
    print("backup saved:", backup.name)

out = old.copy()
out[ID] = out[ID].astype(str).str.strip()
out = out.drop(columns=[c for c in ["title", "description", "time_limit_ms", "memory_limit_kb"] if c in out.columns])
out = out.merge(new, on=ID, how="left")
out.to_parquet(INTERIM / "problems.parquet", index=False)
print("saved problems.parquet:", out.shape, "| columns:", out.columns.tolist())
print(f"Final coverage: {int((out['description'].fillna('').str.len() >= 50).sum())} of {len(out)}")

backup saved: problems_before_descriptions.parquet
saved problems.parquet: (800, 8) | columns: ['problem_id', 'n_chars', 'has_description', 'in_aigcodeset_problem', 'title', 'description', 'time_limit_ms', 'memory_limit_kb']
Final coverage: 800 of 800
